Route elevation analysis

In [ ]:
from pathlib import Path
import pandas as pd

# =========================
# PROJECT PATHS
# =========================

PROJECT_ROOT = Path.cwd().parents[1]
DATA_FEB_DIR = PROJECT_ROOT / "Dataset_feb"
SAMSUNG_DIR = DATA_FEB_DIR / "Dataset_samsung"

routes_file = SAMSUNG_DIR / "Routes.xlsx"
geo_file = SAMSUNG_DIR / "Geo_file.csv"
output_file = SAMSUNG_DIR / "elevation_routes.csv"

print("Routes file found:", routes_file.exists())
print("Geo file found:", geo_file.exists())


# =========================
# LOAD DATA
# =========================

routes_df = pd.read_excel(routes_file)
geo_df = pd.read_csv(geo_file)


# =========================
# CONVERT DATE AND TIME COLUMNS
# =========================

routes_df["start_time"] = pd.to_datetime(routes_df["start_time"])
routes_df["end_time"] = pd.to_datetime(routes_df["end_time"])
geo_df["timestamp"] = pd.to_datetime(geo_df["timestamp"])


# =========================
# CONVERT ALTITUDE TO NUMERIC
# =========================

geo_df["altitude"] = pd.to_numeric(
    geo_df["altitude"],
    errors="coerce"
)


# =========================
# CALCULATE ELEVATION GAIN AND LOSS
# =========================

def calculate_elevation_gain_loss(user_id, start_time, end_time, geo_data):

    # Select GPS points belonging to the participant
    # and occurring during the route
    route_points = geo_data[
        (geo_data["user_id"] == user_id)
        & (geo_data["timestamp"] >= start_time)
        & (geo_data["timestamp"] <= end_time)
    ].copy()

    # Sort GPS points chronologically
    route_points = route_points.sort_values("timestamp")

    # Remove points with missing altitude
    route_points = route_points.dropna(subset=["altitude"])

    # At least two GPS points are required to calculate altitude changes
    if len(route_points) < 2:
        return pd.Series([0, 0, 0])

    # Calculate altitude differences between consecutive GPS points
    route_points["alt_diff"] = route_points["altitude"].diff()

    # Sum positive altitude differences
    elevation_gain = route_points.loc[
        route_points["alt_diff"] > 0,
        "alt_diff"
    ].sum()

    # Sum the absolute value of negative altitude differences
    elevation_loss = route_points.loc[
        route_points["alt_diff"] < 0,
        "alt_diff"
    ].abs().sum()

    # Number of GPS points used for the calculation
    n_geo_points = len(route_points)

    return pd.Series([
        elevation_gain,
        elevation_loss,
        n_geo_points
    ])


# =========================
# APPLY CALCULATION TO EACH ROUTE
# =========================

routes_df[
    ["elevation_gain", "elevation_loss", "n_geo_points"]
] = routes_df.apply(
    lambda row: calculate_elevation_gain_loss(
        row["user_id"],
        row["start_time"],
        row["end_time"],
        geo_df
    ),
    axis=1
)


# =========================
# NET ELEVATION CHANGE
# =========================

routes_df["elevation_net"] = (
    routes_df["elevation_gain"]
    - routes_df["elevation_loss"]
)


# =========================
# ELEVATION PROFILE CLASSIFICATION
# =========================

def classify_elevation_balance(x):
    if x > 0:
        return "more_uphill"
    elif x < 0:
        return "more_downhill"
    else:
        return "balanced"


routes_df["elevation_profile"] = (
    routes_df["elevation_net"]
    .apply(classify_elevation_balance)
)


# =========================
# SAVE RESULTS
# =========================

routes_df.to_csv(
    output_file,
    index=False
)

print("Elevation analysis completed successfully!")
print(f"Saved to: {output_file}")

print(
    routes_df[
        [
            "route_id",
            "user_id",
            "elevation_gain",
            "elevation_loss",
            "elevation_net",
            "elevation_profile",
            "n_geo_points"
        ]
    ].head()
)

Merge with other route features

In [ ]:
# =========================
# LOAD ROUTE FEATURES
# =========================

features_file = SAMSUNG_DIR / "rotas_features.xlsx"
final_output_file = SAMSUNG_DIR / "rotas_features_final.xlsx"

features_df = pd.read_excel(features_file)


# =========================
# SELECT ELEVATION FEATURES
# =========================

elevation_features = routes_df[
    ["route_id", "elevation_gain", "elevation_loss"]
].copy()


# =========================
# MERGE FEATURES
# =========================

rotas_features_final = features_df.merge(
    elevation_features,
    on="route_id",
    how="left"
)


# =========================
# SAVE FINAL ROUTE FEATURE FILE
# =========================

rotas_features_final.to_excel(
    final_output_file,
    index=False
)

print("Final route feature file created successfully!")
print(f"Saved to: {final_output_file}")
print("Shape:", rotas_features_final.shape)
print(rotas_features_final.columns.tolist())